In [ ]:
import os
import pandas as pd
import base64
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output
import gradio as gr

# --- CONFIGURACIÓN DE BASE DE DATOS EXCEL ---
EXCEL_FILE = 'inventario_minimarket.xlsx'

def inicializar_excel():
    if not os.path.exists(EXCEL_FILE):
        columnas = [
            'Fecha y Hora', 'Responsable', 'Nombre', 'Categoría',
            'Marca', 'Proveedor', 'Costo Unitario', 'Cantidad',
            'Precio Venta', 'Stock Mínimo'
        ]
        df = pd.DataFrame(columns=columnas)
        df.to_excel(EXCEL_FILE, index=False)

inicializar_excel()

# --- PROCESAMIENTO DE LA IMAGEN DE LOGO (1.png) ---
img_path = '/content/1.png'
logo_html = ""
if os.path.exists(img_path):
    with open(img_path, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode('utf-8')
    # Generar etiqueta img usando la cadena codificada para asegurar que cargue correctamente en Jupyter
    logo_html = f'<img src="data:image/png;base64,{encoded_string}" style="width: 100%; height: 100%; object-fit: cover; border-radius: 50%;" />'
else:
    # Respaldo simple si no existe el archivo por algún motivo
    logo_html = f'<div style="font-size: 24px;">🏪</div>'

# --- ESTILO GLOBAL Y TÍTULOS ---
estilo_global = widgets.HTML(
    value="""
    <style>
        @import url('https://fonts.cdnfonts.com/css/calibri');
        * {
            font-family: 'Calibri', 'Segoe UI', sans-serif !important;
        }
        .widget-label {
            font-size: 14px !important;
            font-weight: bold !important;
            color: #2b4463 !important;
        }
        .widget-text input, .widget-dropdown select, .widget-number input {
            font-size: 14px !important;
        }
        /* Estilos para la tabla visual de ingresos */
        .tabla-ingresos {
            width: 100%;
            border-collapse: collapse;
            margin-top: 15px;
            font-size: 13px;
        }
        .tabla-ingresos th {
            background-color: #2b4463;
            color: white;
            padding: 8px 10px;
            text-align: left;
            font-weight: bold;
            border: 1px solid #ddd;
        }
        .tabla-ingresos td {
            padding: 8px 10px;
            border: 1px solid #ddd;
            color: #333;
        }
        .tabla-ingresos tr:nth-child(even) {
            background-color: #f8f9fa;
        }
        .tabla-ingresos tr:hover {
            background-color: #f1f3f5;
        }
        /* Estilos para el banner de alerta de stock */
        .alerta-stock {
            background-color: #fff3cd;
            color: #856404;
            border: 1px solid #ffeeba;
            padding: 12px;
            margin-top: 10px;
            border-radius: 5px;
            font-size: 14px;
            font-weight: bold;
        }
    </style>
    """
)

titulo = widgets.HTML(
    value=f"""
    <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 3px solid #2b4463; padding-bottom: 15px;">
        <div style="color: #1a446c;">
            <h2 style="margin: 0px; font-size: 24px; font-weight: bold; letter-spacing: 0.5px;">INVENTARIO PRODUCTOS MINIMARKET</h2>
            <p style="margin: 5px 0px 0px 0px; color: #666; font-size: 13px; font-weight: bold;">FORMULARIO DE REGISTRO DE INGRESO Y EGRESO CON BASE DE DATOS EN EXCEL.</p>
        </div>
        <div style="display: flex; align-items: center; justify-content: center; background: #ffffff; padding: 2px; border-radius: 50%; width: 65px; height: 65px; border: 2.5px solid #2b4463; box-shadow: 0px 4px 6px rgba(0,0,0,0.1); overflow: hidden;">
            {logo_html}
        </div>
    </div>
    """
)

def crear_subtitulo(texto):
    return widgets.HTML(value=f"<h3 style='font-size: 16px; font-weight: bold; margin-top: 15px; margin-bottom: 10px; color: #2b4463; text-transform: uppercase;'><b>{texto}</b></h3>")

# --- SECCIÓN RESPONSABLE (INICIO DE SESIÓN) ---
style_label = {'description_width': '120px'}
layout_input = widgets.Layout(width='340px')

input_responsable = widgets.Text(
    description='Responsable:',
    placeholder='Nombre de quien registra (Ej. Juan Pérez)',
    style=style_label,
    layout=layout_input
)

# --- CAMPOS DE ENTRADA ---
# Sección Registro & Detalles
input_nombre = widgets.Text(description='Nombre:', placeholder='Ej. Arroz Grano Largo', style=style_label, layout=layout_input)
input_categoria = widgets.Dropdown(description='Categoría:', options=['Abarrotes', 'Lácteos', 'Bebidas', 'Limpieza'], value='Abarrotes', style=style_label, layout=layout_input)
input_marca = widgets.Text(description='Marca:', placeholder='Ej. Marca Ideal', style=style_label, layout=layout_input)
input_proveedor = widgets.Text(description='Proveedor:', placeholder='Ej. Distribuidora Sur', style=style_label, layout=layout_input)

# Sección Precios e Inventario
input_costo = widgets.IntText(description='Costo Unit.:', value=0, style=style_label, layout=layout_input)
input_cantidad = widgets.IntText(description='Cantidad:', value=0, style=style_label, layout=layout_input)
input_precio = widgets.IntText(description='Precio Venta:', value=0, style=style_label, layout=layout_input)
input_stock_min = widgets.BoundedIntText(description='Stock Mín:', value=5, min=0, max=100, style=style_label, layout=layout_input)

# --- BOTONES PERSONALIZADOS ---
btn_registrar = widgets.Button(description='Registrar', layout=widgets.Layout(width='160px', height='36px'))
btn_registrar.style.button_color = '#2b4463'
btn_registrar.style.text_color = 'white'

btn_mostrar = widgets.Button(description='Mostrar', layout=widgets.Layout(width='160px', height='36px'))
btn_mostrar.style.button_color = '#7e8d97'
btn_mostrar.style.text_color = 'white'

btn_limpiar = widgets.Button(description='Limpiar', layout=widgets.Layout(width='160px', height='36px'))
btn_limpiar.style.button_color = '#cbd2d9'
btn_limpiar.style.text_color = 'white'

btn_vaciar = widgets.Button(description='Vaciar', layout=widgets.Layout(width='160px', height='36px'))
btn_vaciar.style.button_color = '#b0a3a2'
btn_vaciar.style.text_color = 'white'

btn_descargar = widgets.Button(description='Descargar Excel', layout=widgets.Layout(width='330px', height='38px'))
btn_descargar.style.button_color = '#27ae60'
btn_descargar.style.text_color = 'white'

# --- ÁREA DE SALIDA DE NOTIFICACIONES Y TABLAS ---
out_resultado = widgets.Output()

# --- CONTROLADORES DE EVENTOS (LÓGICA) ---

def actualizar_tabla_visual():
    try:
        if os.path.exists(EXCEL_FILE):
            df = pd.read_excel(EXCEL_FILE)
            if not df.empty:
                # Generar tabla HTML estilizada
                html_table = "<table class='tabla-ingresos'><thead><tr>"
                for col in df.columns:
                    html_table += f"<th>{col}</th>"
                html_table += "</tr></thead><tbody>"
                for _, row in df.iterrows():
                    html_table += "<tr>"
                    for val in row:
                        html_table += f"<td>{val}</td>"
                    html_table += "</tr>"
                html_table += "</tbody></table>"
                return html_table
            else:
                return "<p style='color: #666; font-style: italic; margin-top: 10px;'>La base de datos en Excel está vacía.</p>"
        else:
            return "<p style='color: #666; font-style: italic; margin-top: 10px;'>No se encontró el archivo de base de datos.</p>"
    except Exception as e:
        return f"<p style='color: red;'>Error al cargar la tabla: {e}</p>"

def al_registrar_click(b):
    with out_resultado:
        clear_output()
        responsable = input_responsable.value.strip()
        nombre = input_nombre.value.strip()
        cantidad = input_cantidad.value
        stock_minimo = input_stock_min.value

        if not responsable:
            print("⚠️ Error: Debe ingresar el nombre del Responsable (Inicio de Sesión).")
            return
        if not nombre:
            print("⚠️ Error: El nombre del producto es obligatorio.")
            return

        # Preparar los datos
        nuevo_registro = {
            'Fecha y Hora': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'Responsable': responsable,
            'Nombre': nombre,
            'Categoría': input_categoria.value,
            'Marca': input_marca.value.strip() or 'N/A',
            'Proveedor': input_proveedor.value.strip() or 'N/A',
            'Costo Unitario': input_costo.value,
            'Cantidad': cantidad,
            'Precio Venta': input_precio.value,
            'Stock Mínimo': stock_minimo
        }

        try:
            inicializar_excel()
            df = pd.read_excel(EXCEL_FILE)
            df = pd.concat([df, pd.DataFrame([nuevo_registro])], ignore_index=True)
            df.to_excel(EXCEL_FILE, index=False)

            print(f"✅ ¡Producto '{nombre}' registrado exitosamente por {responsable} en Excel!")

            # Alerta de Stock Mínimo
            if cantidad < stock_minimo:
                alerta_html = f"""
                <div class='alerta-stock'>
                    ⚠️ ¡ALERTA DE STOCK BAJO! El producto registrado '{nombre}' posee una cantidad actual de {cantidad},
                    la cual es menor al stock mínimo establecido de {stock_minimo}.
                </div>
                """
                display(widgets.HTML(alerta_html))

            # Limpiar campos de producto
            al_limpiar_click(None)
            # Mostrar tabla actualizada automáticamente
            display(widgets.HTML(actualizar_tabla_visual()))
        except Exception as e:
            print(f"❌ Error al escribir en el archivo de Excel: {e}")

def al_mostrar_click(b):
    with out_resultado:
        clear_output()
        display(widgets.HTML("<h4 style='color: #2b4463; font-weight: bold; margin-top: 10px;'>TABLA DE INGRESOS REGISTRADOS:</h4>"))
        display(widgets.HTML(actualizar_tabla_visual()))

def al_limpiar_click(b):
    input_nombre.value = ''
    input_categoria.value = 'Abarrotes'
    input_marca.value = ''
    input_proveedor.value = ''
    input_costo.value = 0
    input_cantidad.value = 0
    input_precio.value = 0
    input_stock_min.value = 5
    if b is not None:
        with out_resultado:
            clear_output()
            print("🧹 Formulario de producto restablecido.")

def al_descargar_click(b):
    with out_resultado:
        clear_output()
        if os.path.exists(EXCEL_FILE):
            try:
                with open(EXCEL_FILE, "rb") as f:
                    excel_data = f.read()
                b64 = base64.b64encode(excel_data).decode('utf-8')
                filename = "inventario_minimarket.xlsx"
                # Crear un enlace HTML de descarga automática
                dl_link = f"""
                <div style="margin-top: 10px; padding: 10px; background-color: #e8f5e9; border: 1px solid #c8e6c9; border-radius: 5px;">
                    <p style="color: #2e7d32; font-weight: bold; margin: 0 0 8px 0;">📥 El archivo Excel está listo para ser descargado:</p>
                    <a href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64}" download="{filename}"
                       style="display: inline-block; background-color: #27ae60; color: white; padding: 8px 15px; text-decoration: none; border-radius: 4px; font-weight: bold;">
                       Haz Clic Aquí para Descargar
                    </a>
                </div>
                """
                display(widgets.HTML(dl_link))
            except Exception as e:
                print(f"❌ Error al generar la descarga: {e}")
        else:
            print("⚠️ Error: No se ha encontrado el archivo Excel del inventario para descargar.")

def al_vaciar_click(b):
    with out_resultado:
        clear_output()
        try:
            if os.path.exists(EXCEL_FILE):
                os.remove(EXCEL_FILE)
                inicializar_excel()
                print("🗑️ Base de datos de Excel eliminada y reiniciada con éxito.")
                display(widgets.HTML(actualizar_tabla_visual()))
            else:
                print("📝 No hay base de datos que eliminar.")
        except Exception as e:
            print(f"❌ Error al vaciar la base de datos: {e}")

# Enlace de funciones a eventos
btn_registrar.on_click(al_registrar_click)
btn_mostrar.on_click(al_mostrar_click)
btn_limpiar.on_click(al_limpiar_click)
btn_descargar.on_click(al_descargar_click)
btn_vaciar.on_click(al_vaciar_click)

# --- ESTRUCTURACIÓN DEL DISEÑO (GRID & BOXES) ---

seccion_sesion = widgets.VBox([
    crear_subtitulo('Inicio de Sesión (Responsable)'),
    input_responsable
])

columna_izquierda_1 = widgets.VBox([crear_subtitulo('Registro'), input_nombre, input_categoria])
columna_derecha_1 = widgets.VBox([crear_subtitulo('Detalles'), input_marca, input_proveedor])
bloque_1 = widgets.HBox([columna_izquierda_1, columna_derecha_1], layout=widgets.Layout(grid_gap='40px'))

columna_izquierda_2 = widgets.VBox([input_costo, input_cantidad])
columna_derecha_2 = widgets.VBox([input_precio, input_stock_min])
bloque_2 = widgets.HBox([columna_izquierda_2, columna_derecha_2], layout=widgets.Layout(grid_gap='40px'))

linea_divisoria = widgets.HTML(value="<hr style='border: 0; border-top: 1px solid #ccc; margin-top: 20px; margin-bottom: 15px;'>")

# Cuatro botones grises de acción en la primera fila
fila_acciones_gris = widgets.HBox(
    [btn_registrar, btn_mostrar, btn_limpiar, btn_vaciar],
    layout=widgets.Layout(grid_gap='12px', margin='5px 0px')
)

# Botón verde de descarga centrado debajo
fila_descarga_excel = widgets.HBox(
    [btn_descargar],
    layout=widgets.Layout(justify_content='center', margin='10px 0px')
)

# --- ENSAMBLAJE FINAL DE LA INTERFAZ ---
formulario_completo = widgets.VBox([
    estilo_global,
    titulo,
    seccion_sesion,
    bloque_1,
    crear_subtitulo('Precios e Inventario'),
    bloque_2,
    linea_divisoria,
    fila_acciones_gris,
    fila_descarga_excel,
    out_resultado
], layout=widgets.Layout(padding='15px', max_width='780px', border='1px solid #ddd', border_radius='8px'))

# Renderizar en la celda de Jupyter
display(formulario_completo)

In [ ]:
!pip install --upgrade gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 5.7 MB/s eta 0:00:00
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 2.7.1
    Uninstalling gradio_client-2.7.1:
      Successfully uninstalled gradio_client-2.7.1
  Attempting uninstall: gradio
    Found existing installation: gradio 6.29.0
    Uninstalling gradio-6.29.0:
      Successfully uninstalled gradio-6.29.0
